In [1]:
import os 
import sys
import warnings
import subprocess
import pandas as pd
import numpy as np
import pybedtools
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from gene2probe import *

This tutorial guides you through the design of custom probes against a gene of interest.

First, we need to specify the gene symbol, and which feature we are interested in designing probes agains (in this case we are using exons).
We also need to specify an output directory for the analysis.

### 1. Specify parameters

In [2]:
## Specify gene of interest and feature of interest
gene_ID = 'MIR4435-2HG'
mode = 'transcript' ## Whether   to consider only exons / introns or full gene

## specify output directory
out_dir = '../sample_run/probeDesign_' + gene_ID + '_' + mode + '/'
## Create output directory
os.makedirs(out_dir, exist_ok=True)

Additionally, we need to provide the path to several resource files. Many of these files can be obtained from [UCSC table browser](https://genome.ucsc.edu/cgi-bin/hgTables).

We also need a blast database, such as the one we generated in the [previous tutorial](https://github.com/Teichlab/gene2probe/blob/main/notebooks/001_make_blast_database.ipynb).

In [3]:
## Required resources (most can be downloaded from 
gtf = '../hg38_resources/hg38.ncbiRefSeq.gtf' ## Gene annotation in gtf file
## We recommend using RefSeq as this is manually curated and more likely to contain an isoform that is present across most cell types
## Alternatively, one can filter based on RNA-seq data for a cell type/tissue of interest
fasta = '../hg38_resources/hg38.fa' ## Genome in fasta file
snp_db = '../hg38_resources/hg38_snp151Common.bed' ## Database of known SNPs and small indels
repeats = '../hg38_resources/hg38_rmsk.bed' ## bed file with repeats/low complexity regions to be excluded
gaps = '../hg38_resources/hg38_rmsk.bed' ## bed file with gaps in the genome assembly to be excluded
blast_db = '../hg38_resources/001_blastdb/hg38_ncbiRefSeq_transcripts_db' ## Database of all human transcripts to blast against

In [4]:
## Path to blast binaries.
## Replace with your conda environment
## This can also be omitted if you started the jupyter session from within the gene2probe conda environment
print('current working directory:', os.getcwd())

# blast_exec_path = f"{os.environ['HOME']}/.miniforge3/envs/gene2probe_env/bin/"
blast_exec_path = "/opt/homebrew/Caskroom/miniconda/base/envs/gene2probe_env/bin/"

if not os.path.isdir(blast_exec_path):
    warnings.warn(
        f'BLAST executable directory not found: {blast_exec_path}',
        RuntimeWarning,
    )

print('blast executable path:', blast_exec_path)

current working directory: /Users/base/gene2probe/notebooks
blast executable path: /opt/homebrew/Caskroom/miniconda/base/envs/gene2probe_env/bin/


Finally, we need to provide a set of parameters related to our probe's length, at which nucleotide it's split (if at all), the acceptable range for GC content and any specific requirements for individual nucleotides.

Here we are following the [recommendations of 10x Genomics for custom probes for VisiumHD/VisiumFFPE/Flex](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

In [5]:
## Additional parameters regarding how the probe should look like
probe_length = 50 ## Length of probe in nucleotides
split_nt = 25 ## Index of nucleotide to split the probe at (start of RHS) - set to None if splitting probe is not needed
min_GC = 0.44 ## Minimum GC content for probe (if split probe, applied to both LHS and RHS)
max_GC = 0.72 ## Maximum GC content for probe (if split probe, applied to both LHS and RHS)
required_nts = {24: 'T'} ## Dictionary of index (0-based) for required nts - by default, 25th nucleotide must be a T - set to None if no requirements
probe_offset = 1000 ## Minimum distance between probes - 10 bp is the recommended minimum by 10x, this can also be adjusted depending on how many probes pass other cutoffs
n_desired_probes =3 ## Number of probes to be designed.
min_mismatches = 5 ## Minimum number of mismatches (in at least LHS or RHS) - here we require in both to be more conservative

In [6]:
## Optionally, we can also specify adapters that have to be added to the probes.
## For example, for visiumHD:
LHS_pref = 'CCTTGGCACCCGAGAATTCCA' ## Will be added to the 5' of the LHS probe
LHS_suff = '' ## Will be added to the 3' of the LHS probe
RHS_pref = '/5Phos/' ## Will be added to the 5' of the RHS probe
RHS_suff = 'CCCATATAAGAAA' ## Will be added to the 3' of the RHS probe

## Leave as empty strings if you don't want to use them


### 2. Generate k-mers

Now we can start by reading the gene annotation and filtering for our gene of interest.

In [7]:
## Read gtf file
gene_anno = read_gtf(gtf)

In [8]:
gene_anno

,seqname,source,feature,start,end,score,strand,frame,attribute
0,chrM,ncbiRefSeq.2022-10-28,transcript,15956,16023,.,-,.,"gene_id ""TRNP""; transcript_id ""rna-TRNP""; gen..."
1,chrM,ncbiRefSeq.2022-10-28,exon,15956,16023,.,-,.,"gene_id ""TRNP""; transcript_id ""rna-TRNP""; exon..."
2,chrM,ncbiRefSeq.2022-10-28,transcript,15888,15953,.,+,.,"gene_id ""TRNT""; transcript_id ""rna-TRNT""; gen..."
3,chrM,ncbiRefSeq.2022-10-28,exon,15888,15953,.,+,.,"gene_id ""TRNT""; transcript_id ""rna-TRNT""; exon..."
4,chrM,ncbiRefSeq.2022-10-28,transcript,14747,15887,.,+,.,"gene_id ""CYTB""; transcript_id ""rna-CYTB""; gen..."
...,...,...,...,...,...,...,...,...,...
4886697,chr1,ncbiRefSeq.2022-10-28,exon,29321,29370,.,-,.,"gene_id ""WASH7P""; transcript_id ""NR_024540.1"";..."
4886698,chr1,ncbiRefSeq.2022-10-28,transcript,11874,14409,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."
4886699,chr1,ncbiRefSeq.2022-10-28,exon,11874,12227,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."
4886700,chr1,ncbiRefSeq.2022-10-28,exon,12613,12721,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."


In [9]:
## Extract regions corresponding to gene of interest (symbol: gene_name, Ensembl ID: gene_ID), subset to feature of interest and convert to bed style dataframe:
roi_bed = get_region_of_interest(gene_anno, gene_ID, gene_id_type = 'gene_name', feature=mode)

In [10]:
roi_bed

,seqname,start,end,name,score,strand
0,chr2,111429308,111495161,MIR4435-2HG_0,.,-
1,chr2,111207772,111495161,MIR4435-2HG_1,.,-
2,chr2,111207772,111495161,MIR4435-2HG_2,.,-
3,chr2,111207772,111495161,MIR4435-2HG_3,.,-
4,chr2,111207772,111495161,MIR4435-2HG_4,.,-
5,chr2,111195865,111495161,MIR4435-2HG_5,.,-
6,chr2,111195865,111495161,MIR4435-2HG_6,.,-
7,chr2,111195865,111495161,MIR4435-2HG_7,.,-


After extracting the coordinates of interest and converting to a bed-like format, we can generate all possible kmers that fall within these regions.

In [11]:
kmers = generate_kmers(roi_bed, k=probe_length)

In [12]:
kmers

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
2112900,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
2112901,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
2112902,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
2112903,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


In [13]:
## Export unfiltered
kmers.to_csv((out_dir + 'kmers_all.csv'))

### 3. Exclude annotated repeats/polymorphism

We can next exclude kmers overlapping undesired regions (repeats, low complexity regions, common polymorphism, gaps in the assembly) from further consideration.

Ideally, we will exclude everything that overlaps a repeat or polymorphism, but if we only have too few kmers available, we might need to relax these requirements (e.g., to only exclude kmers overlapping SNPs around the ligation junction, if the probes are split).

In [14]:
## For example, we could have removed all kmers overlapping a repeat/low complexity region within 5 nts of the ligation junction:
remove_overlaps(kmers, repeats, core=[20,30])

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
1239845,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
1239846,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
1239847,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
1239848,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


In [15]:
## In this case we have a lot of possible kmers, so we will remove those with overlaps in any part of the probe:
kmers = remove_overlaps(kmers, repeats)

In [16]:
kmers 

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
1198863,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
1198864,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
1198865,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
1198866,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


In [17]:
## Doing the same for gaps in the assembly (very unlikely since we are starting with annotated exons)
kmers = remove_overlaps(kmers, gaps)

In [18]:
## And more importantly, against common polymorphism (SNPs, short indels)
kmers = remove_overlaps(kmers, snp_db)

In [19]:
kmers

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
1028172,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
1028173,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
1028174,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
1028175,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


### 4. Filter for desirable sequence features

Having excluded undesirable kmers based on intersection with genomic annotations, the next step is to consider their sequence features.

For this, we first extract the sequences of each k-mer, and then estimate features such as GC content and the presence of desired nucleotides in specific positions.

In [20]:
## Get DNA for the transcript
kmers_bed = pybedtools.BedTool.from_dataframe(kmers)
kmers_seq = kmers_bed.sequence(fi=fasta, s=True) 

## We can read in the sequences and simultaneously monitor GC content and count the longest homopolymer stretch
kmers_seq_stats = get_sequence_stats(kmers_seq.seqfn, probe_length, split_nt)

In [21]:
## Combining with our dataframe
kmers = pd.merge(kmers, kmers_seq_stats, left_index=True, right_index=True)

In [22]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-,chr2:111429308-111429358(-),GTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGAAAAC...,TTCTGTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAG...,0.32,4,0.24,0.40
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-,chr2:111429309-111429359(-),TGTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGAAAA...,TCTGTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAGA...,0.32,4,0.24,0.40
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-,chr2:111429310-111429360(-),GTGTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGAAA...,CTGTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAGAC...,0.34,4,0.24,0.44
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-,chr2:111429311-111429361(-),TGTGTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGAA...,TGTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAGACA...,0.32,4,0.20,0.44
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-,chr2:111429312-111429362(-),CTGTGTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGA...,GTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAGACAC...,0.34,4,0.24,0.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1028172,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-,chr2:111495107-111495157(-),GTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCA...,CTCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTC...,0.50,3,0.52,0.48
1028173,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-,chr2:111495108-111495158(-),AGTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGC...,TCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTCA...,0.48,3,0.48,0.48
1028174,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-,chr2:111495109-111495159(-),AAGTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAG...,CATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTCAT...,0.48,3,0.52,0.44
1028175,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-,chr2:111495110-111495160(-),CAAGTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCA...,ATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTCATA...,0.48,3,0.52,0.44


In [23]:
## Check for required nucleotides in specific positions:
if required_nts is not None:
    kmers['has_required_nts'] = check_for_required_nts(kmers, required_nts)
    print(kmers['has_required_nts'].value_counts())
    ## Filter for required nucleotides
    kmers = kmers[kmers['has_required_nts']==True].reset_index(drop=True)

has_required_nts
False    760980
True     267197
Name: count, dtype: int64


In [24]:
## Export kmers before filtering
kmers.to_csv((out_dir + 'kmers_candidates_unfiltered.csv'))

In [25]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-,chr2:111429308-111429358(-),GTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGAAAAC...,TTCTGTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAG...,0.32,4,0.24,0.40,True
1,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-,chr2:111429311-111429361(-),TGTGTGTCTTAATCCCTTGTCCTTCATTAAAAGCAAAACTAAAGAA...,TGTTTTCTTTAGTTTTGCTTTTAATGAAGGACAAGGGATTAAGACA...,0.32,4,0.20,0.44,True
2,chr2,111429325,111429375,MIR4435-2HG_0_17,.,-,chr2:111429325-111429375(-),GTCTGGCCAGTCTCTGTGTGTCTTAATCCCTTGTCCTTCATTAAAA...,TTGCTTTTAATGAAGGACAAGGGATTAAGACACACAGAGACTGGCC...,0.44,4,0.36,0.52,True
3,chr2,111429326,111429376,MIR4435-2HG_0_18,.,-,chr2:111429326-111429376(-),TGTCTGGCCAGTCTCTGTGTGTCTTAATCCCTTGTCCTTCATTAAA...,TGCTTTTAATGAAGGACAAGGGATTAAGACACACAGAGACTGGCCA...,0.44,4,0.36,0.52,True
4,chr2,111429342,111429392,MIR4435-2HG_0_34,.,-,chr2:111429342-111429392(-),TGGTCGGTTTCCCATTTGTCTGGCCAGTCTCTGTGTGTCTTAATCC...,CAAGGGATTAAGACACACAGAGACTGGCCAGACAAATGGGAAACCG...,0.50,3,0.44,0.56,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267192,chr2,111495094,111495144,MIR4435-2HG_7_299229,.,-,chr2:111495094-111495144(-),GTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCATGAGTCATCTCGT...,TGGAACGAGATGACTCATGCTGACTCATAGCCACCACTTCCTCTCC...,0.54,3,0.48,0.60,True
267193,chr2,111495097,111495147,MIR4435-2HG_7_299232,.,-,chr2:111495097-111495147(-),AATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCATGAGTCATCT...,AACGAGATGACTCATGCTGACTCATAGCCACCACTTCCTCTCCCGA...,0.50,3,0.44,0.56,True
267194,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-,chr2:111495107-111495157(-),GTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCA...,CTCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTC...,0.50,3,0.52,0.48,True
267195,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-,chr2:111495108-111495158(-),AGTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGC...,TCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTCA...,0.48,3,0.48,0.48,True


In [26]:
## Filter for GC content
kmers = filter_by_GC_content(kmers, min_GC, max_GC)

In [27]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr2,111429354,111429404,MIR4435-2HG_0_46,.,-,chr2:111429354-111429404(-),CATGGGCTGGTCTGGTCGGTTTCCCATTTGTCTGGCCAGTCTCTGT...,ACACACAGAGACTGGCCAGACAAATGGGAAACCGACCAGACCAGCC...,0.56,3,0.48,0.64,True
1,chr2,111429378,111429428,MIR4435-2HG_0_70,.,-,chr2:111429378-111429428(-),GTGGTCTGCCTGTGATATTTTGGTCATGGGCTGGTCTGGTCGGTTT...,TGGGAAACCGACCAGACCAGCCCATGACCAAAATATCACAGGCAGA...,0.54,4,0.60,0.48,True
2,chr2,111429387,111429437,MIR4435-2HG_0_79,.,-,chr2:111429387-111429437(-),CATTTGCGGGTGGTCTGCCTGTGATATTTTGGTCATGGGCTGGTCT...,GACCAGACCAGCCCATGACCAAAATATCACAGGCAGACCACCCGCA...,0.54,4,0.52,0.56,True
3,chr2,111429389,111429439,MIR4435-2HG_0_81,.,-,chr2:111429389-111429439(-),TGCATTTGCGGGTGGTCTGCCTGTGATATTTTGGTCATGGGCTGGT...,CCAGACCAGCCCATGACCAAAATATCACAGGCAGACCACCCGCAAA...,0.54,4,0.48,0.60,True
4,chr2,111429411,111429461,MIR4435-2HG_0_103,.,-,chr2:111429411-111429461(-),CCACTGTGGACTCTGAGGCCTCTGCATTTGCGGGTGGTCTGCCTGT...,TATCACAGGCAGACCACCCGCAAATGCAGAGGCCTCAGAGTCCACA...,0.58,3,0.52,0.64,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72259,chr2,111495069,111495119,MIR4435-2HG_7_299204,.,-,chr2:111495069-111495119(-),AGTCAGCATGAGTCATCTCGTTCCAATGAGAATGAAGGCTGAGGTG...,CGCACACCTCAGCCTTCATTCTCATTGGAACGAGATGACTCATGCT...,0.50,2,0.52,0.48,True
72260,chr2,111495070,111495120,MIR4435-2HG_7_299205,.,-,chr2:111495070-111495120(-),GAGTCAGCATGAGTCATCTCGTTCCAATGAGAATGAAGGCTGAGGT...,GCACACCTCAGCCTTCATTCTCATTGGAACGAGATGACTCATGCTG...,0.50,2,0.48,0.52,True
72261,chr2,111495094,111495144,MIR4435-2HG_7_299229,.,-,chr2:111495094-111495144(-),GTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCATGAGTCATCTCGT...,TGGAACGAGATGACTCATGCTGACTCATAGCCACCACTTCCTCTCC...,0.54,3,0.48,0.60,True
72262,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-,chr2:111495107-111495157(-),GTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCA...,CTCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTC...,0.50,3,0.52,0.48,True


In [28]:
## Candidate kmers
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr2,111429354,111429404,MIR4435-2HG_0_46,.,-,chr2:111429354-111429404(-),CATGGGCTGGTCTGGTCGGTTTCCCATTTGTCTGGCCAGTCTCTGT...,ACACACAGAGACTGGCCAGACAAATGGGAAACCGACCAGACCAGCC...,0.56,3,0.48,0.64,True
1,chr2,111429378,111429428,MIR4435-2HG_0_70,.,-,chr2:111429378-111429428(-),GTGGTCTGCCTGTGATATTTTGGTCATGGGCTGGTCTGGTCGGTTT...,TGGGAAACCGACCAGACCAGCCCATGACCAAAATATCACAGGCAGA...,0.54,4,0.60,0.48,True
2,chr2,111429387,111429437,MIR4435-2HG_0_79,.,-,chr2:111429387-111429437(-),CATTTGCGGGTGGTCTGCCTGTGATATTTTGGTCATGGGCTGGTCT...,GACCAGACCAGCCCATGACCAAAATATCACAGGCAGACCACCCGCA...,0.54,4,0.52,0.56,True
3,chr2,111429389,111429439,MIR4435-2HG_0_81,.,-,chr2:111429389-111429439(-),TGCATTTGCGGGTGGTCTGCCTGTGATATTTTGGTCATGGGCTGGT...,CCAGACCAGCCCATGACCAAAATATCACAGGCAGACCACCCGCAAA...,0.54,4,0.48,0.60,True
4,chr2,111429411,111429461,MIR4435-2HG_0_103,.,-,chr2:111429411-111429461(-),CCACTGTGGACTCTGAGGCCTCTGCATTTGCGGGTGGTCTGCCTGT...,TATCACAGGCAGACCACCCGCAAATGCAGAGGCCTCAGAGTCCACA...,0.58,3,0.52,0.64,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72259,chr2,111495069,111495119,MIR4435-2HG_7_299204,.,-,chr2:111495069-111495119(-),AGTCAGCATGAGTCATCTCGTTCCAATGAGAATGAAGGCTGAGGTG...,CGCACACCTCAGCCTTCATTCTCATTGGAACGAGATGACTCATGCT...,0.50,2,0.52,0.48,True
72260,chr2,111495070,111495120,MIR4435-2HG_7_299205,.,-,chr2:111495070-111495120(-),GAGTCAGCATGAGTCATCTCGTTCCAATGAGAATGAAGGCTGAGGT...,GCACACCTCAGCCTTCATTCTCATTGGAACGAGATGACTCATGCTG...,0.50,2,0.48,0.52,True
72261,chr2,111495094,111495144,MIR4435-2HG_7_299229,.,-,chr2:111495094-111495144(-),GTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCATGAGTCATCTCGT...,TGGAACGAGATGACTCATGCTGACTCATAGCCACCACTTCCTCTCC...,0.54,3,0.48,0.60,True
72262,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-,chr2:111495107-111495157(-),GTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCA...,CTCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTC...,0.50,3,0.52,0.48,True


In [29]:
kmers.to_csv((out_dir + 'kmers_candidates_filtered.csv'))

### 5. Remove probes with potential off-targets

Having identified a set of kmers that fulfill our sequence requirements, we can next proceed with testing whether they are specific to our transcript/exon of interest.

For this we rely on using BLAST. 

We recommend blasting against transcripts (i.e., exons and introns combined) to be as conservative as possible in terms of off-targets. 
However, in cases where it is not possible to obtain enough suitable kmers (e.g., for  short transcripts), it is reasonable to relax this requirement by BLASTing against exons only (a much smaller search space).

At this step, we also want to consider whether our probes are split (as in the current specifications for VisiumHD) or a single oligo. If probes are split, it's best to BLAST each side separately, to make sure that both sides are specific.

In [30]:
## The first thing to do is to export our sequences in fasta format, so that we can use them for BLAST
write_fasta(kmers['name'], kmers['transcript_seq'], (out_dir + 'kmers_candidates_filtered_transcript_seqs.fa'))
## If our probes are meant to be split, we should additionally blast them separately 
## Note that the LHS/RHS in the transcript are reversed compared to the probe (i.e., the LHS of the transcript is complementary to the RHS of the probe)
if split_nt is not None: 
    ## Make split probes
    kmers['transcript_seq_LHS'] = [seq[0:split_nt] for seq in kmers['transcript_seq']]
    kmers['transcript_seq_RHS'] = [seq[split_nt: probe_length] for seq in kmers['transcript_seq']]

    ## We are exporting the transcript sequence as that's the one that has to be blasted against the human transcriptome
    write_fasta(kmers['name'], kmers['transcript_seq_LHS'], (out_dir + 'kmers_candidates_filtered_transcript_seqs_LHS.fa'))
    write_fasta(kmers['name'], kmers['transcript_seq_RHS'], (out_dir + 'kmers_candidates_filtered_transcript_seqs_RHS.fa'))    

In [31]:
blast_res = {}
## First, blast the full probe
blast_res['full'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs.fa'),
                              blastdb = blast_db,
                              path2blastn=(blast_exec_path + 'blastn'),
                              outfile = (out_dir + 'kmers_candidates_filtered_blast_output.txt'))

## Additionally, if probe is split, blast each side separately
if split_nt is not None: 
    blast_res['LHS'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs_LHS.fa'),
                                     blastdb = blast_db,
                                     path2blastn=(blast_exec_path + 'blastn'),
                                     outfile = (out_dir + 'kmers_candidates_filtered_blast_output_LHS.txt'))
    blast_res['RHS'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs_RHS.fa'),
                                     blastdb = blast_db,
                                     path2blastn=(blast_exec_path + 'blastn'),
                                     outfile = (out_dir + 'kmers_candidates_filtered_blast_output_RHS.txt'))

In [32]:
for k in blast_res.keys():
    print(("The following genes were detected in mode: " +  k))
    print(blast_res[k]['sgeneid'].value_counts().head(10))

The following genes were detected in mode: full
sgeneid
MIR4435-2HG     743984
CYTOR            90376
LOC124907867     35798
PTK2              9800
COL21A1           8232
CACNA1C           5152
ABLIM2            4704
ELN               4400
RAP1GAP           4235
RBFOX3            4046
Name: count, dtype: int64
The following genes were detected in mode: LHS
sgeneid
MIR4435-2HG     729611
CYTOR            89891
LOC124907867     33873
PTK2              5600
RAP1GAP           3388
ABLIM2            2982
NRXN3             1924
CELF4             1827
RBFOX1            1800
AUTS2             1785
Name: count, dtype: int64
The following genes were detected in mode: RHS
sgeneid
MIR4435-2HG     735379
CYTOR            90064
LOC124907867     34720
ABLIM2            4704
CELF4             2702
COL21A1           2156
RBFOX3            2146
NRXN3             1851
ANK2              1848
ELN               1760
Name: count, dtype: int64


In [33]:
blast_res['full']

,name,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,sgeneid
0,MIR4435-2HG_0_46,MIR4435-2HG::chr2:111195865-111495161(-),100.0,50,0,0,1,50,65758,65807,8.400000e-17,91.5,MIR4435-2HG
1,MIR4435-2HG_0_46,MIR4435-2HG::chr2:111195865-111495161(-),100.0,50,0,0,1,50,65758,65807,8.400000e-17,91.5,MIR4435-2HG
2,MIR4435-2HG_0_46,MIR4435-2HG::chr2:111195865-111495161(-),100.0,50,0,0,1,50,65758,65807,8.400000e-17,91.5,MIR4435-2HG
3,MIR4435-2HG_0_46,MIR4435-2HG::chr2:111207772-111495161(-),100.0,50,0,0,1,50,65758,65807,8.400000e-17,91.5,MIR4435-2HG
4,MIR4435-2HG_0_46,MIR4435-2HG::chr2:111207772-111495161(-),100.0,50,0,0,1,50,65758,65807,8.400000e-17,91.5,MIR4435-2HG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150399,MIR4435-2HG_7_299243,MIR4435-2HG::chr2:111207772-111495161(-),100.0,50,0,0,1,50,4,53,8.400000e-17,91.5,MIR4435-2HG
1150400,MIR4435-2HG_7_299243,MIR4435-2HG::chr2:111207772-111495161(-),100.0,50,0,0,1,50,4,53,8.400000e-17,91.5,MIR4435-2HG
1150401,MIR4435-2HG_7_299243,MIR4435-2HG::chr2:111207772-111495161(-),100.0,50,0,0,1,50,4,53,8.400000e-17,91.5,MIR4435-2HG
1150402,MIR4435-2HG_7_299243,MIR4435-2HG::chr2:111207772-111495161(-),100.0,50,0,0,1,50,4,53,8.400000e-17,91.5,MIR4435-2HG


Not all BLAST hits will be off-targets. Hopefully, our gene of interest is included in the BLAST output. We therefore need to filter for hits with different gene IDs.

In [34]:
offtargets = []
for k in blast_res.keys():
    offtargets += (detect_offtargets(blast_res[k], gene_ID, min_mismatches=min_mismatches))
## Remove redundancies
offtargets = list(set(offtargets))

In [35]:
len(offtargets)

43075

In [36]:
## Remove off-targets
kmers = kmers[kmers['name'].isin(offtargets)==False].reset_index(drop=True)

In [37]:
kmers 

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS
0,chr2,111495107,111495157,MIR4435-2HG_0_65799,.,-,chr2:111495107-111495157(-),GTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCA...,CTCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTC...,0.50,3,0.52,0.48,True,GTATGAAGAGAATGTCGGGAGAGGA,AGTGGTGGCTATGAGTCAGCATGAG
1,chr2,111495108,111495158,MIR4435-2HG_0_65800,.,-,chr2:111495108-111495158(-),AGTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGC...,TCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTCA...,0.48,3,0.48,0.48,True,AGTATGAAGAGAATGTCGGGAGAGG,AAGTGGTGGCTATGAGTCAGCATGA
2,chr2,111208056,111208106,MIR4435-2HG_1_284,.,-,chr2:111208056-111208106(-),ACAAGCCTGGACTTTGCTCTACAACAGGGTTTTGCATAGGGAGTGG...,CATACCACTCCCTATGCAAAACCCTGTTGTAGAGCAAAGTCCAGGC...,0.48,4,0.48,0.48,True,ACAAGCCTGGACTTTGCTCTACAAC,AGGGTTTTGCATAGGGAGTGGTATG
3,chr2,111208058,111208108,MIR4435-2HG_1_286,.,-,chr2:111208058-111208108(-),CAACAAGCCTGGACTTTGCTCTACAACAGGGTTTTGCATAGGGAGT...,TACCACTCCCTATGCAAAACCCTGTTGTAGAGCAAAGTCCAGGCTT...,0.48,4,0.48,0.48,True,CAACAAGCCTGGACTTTGCTCTACA,ACAGGGTTTTGCATAGGGAGTGGTA
4,chr2,111208059,111208109,MIR4435-2HG_1_287,.,-,chr2:111208059-111208109(-),GCAACAAGCCTGGACTTTGCTCTACAACAGGGTTTTGCATAGGGAG...,ACCACTCCCTATGCAAAACCCTGTTGTAGAGCAAAGTCCAGGCTTG...,0.50,4,0.48,0.52,True,GCAACAAGCCTGGACTTTGCTCTAC,AACAGGGTTTTGCATAGGGAGTGGT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29184,chr2,111429157,111429207,MIR4435-2HG_7_233292,.,-,chr2:111429157-111429207(-),TGTGTGTTGGGCTTGACTCTCTCATAACTCAGCCACGCTTGTCTTG...,GCAGCAAGACAAGCGTGGCTGAGTTATGAGAGAGTCAAGCCCAACA...,0.52,3,0.56,0.48,True,TGTGTGTTGGGCTTGACTCTCTCAT,AACTCAGCCACGCTTGTCTTGCTGC
29185,chr2,111429159,111429209,MIR4435-2HG_7_233294,.,-,chr2:111429159-111429209(-),AATGTGTGTTGGGCTTGACTCTCTCATAACTCAGCCACGCTTGTCT...,AGCAAGACAAGCGTGGCTGAGTTATGAGAGAGTCAAGCCCAACACA...,0.48,3,0.48,0.48,True,AATGTGTGTTGGGCTTGACTCTCTC,ATAACTCAGCCACGCTTGTCTTGCT
29186,chr2,111429213,111429263,MIR4435-2HG_7_233348,.,-,chr2:111429213-111429263(-),GAATAGTGTTTGAGGGAACCCTGGCACGTCCACACAGCTCATCTGA...,CATGTCAGATGAGCTGTGTGGACGTGCCAGGGTTCCCTCAAACACT...,0.52,3,0.52,0.52,True,GAATAGTGTTTGAGGGAACCCTGGC,ACGTCCACACAGCTCATCTGACATG
29187,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-,chr2:111495107-111495157(-),GTATGAAGAGAATGTCGGGAGAGGAAGTGGTGGCTATGAGTCAGCA...,CTCATGCTGACTCATAGCCACCACTTCCTCTCCCGACATTCTCTTC...,0.50,3,0.52,0.48,True,GTATGAAGAGAATGTCGGGAGAGGA,AGTGGTGGCTATGAGTCAGCATGAG


### 6. Select non-overlapping probes

At this point, we have effectively acquired a set of usable probes. They don't overlap undesirable regions (repeats/polymorphism), have desirable sequence features (GC content, specific nucleotides) and are specific to our gene of interest.

In this particular case, we still have a lot of possible k-mers (much more than the number of probes we intend to design). We can therefore choose to prioritise k-mers with shorter homopolymer stretches, as these are also discouraged by the [10x recommendations](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

However, at this stage you might want to consider ranking probes in a diffferent way, depending on your application.

After having ranked our k-mers in whatever way we think is reasonable at this stage, we can proceed with selecting the top probe, then removing all overlapping/adjacent probes (within a window determined by `probe_offset`).

Since we have so many available probes, we will be increasing `probe_offset` from `100 (default)` to `1000`.

In [38]:
## Sort in increasing homopolymer length
kmers = kmers.sort_values('longest_homopolymer', ascending=True).reset_index(drop=True)

In [39]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS
0,chr2,111414483,111414533,MIR4435-2HG_2_206711,.,-,chr2:111414483-111414533(-),CCTGAGTTGCTGCCAACCTTACAATATCCAGCTCCGTCCTCAGAGC...,TGTGGCTCTGAGGACGGAGCTGGATATTGTAAGGTTGGCAGCAACT...,0.54,2,0.60,0.48,True,CCTGAGTTGCTGCCAACCTTACAAT,ATCCAGCTCCGTCCTCAGAGCCACA
1,chr2,111315351,111315401,MIR4435-2HG_1_107579,.,-,chr2:111315351-111315401(-),GCTCTCCTGTTACTTCCTGTGCTTAATTAAGTGGCTCTGCAGGTGA...,TGAGTCACCTGCAGAGCCACTTAATTAAGCACAGGAAGTAACAGGA...,0.48,2,0.48,0.48,True,GCTCTCCTGTTACTTCCTGTGCTTA,ATTAAGTGGCTCTGCAGGTGACTCA
2,chr2,111348488,111348538,MIR4435-2HG_2_140716,.,-,chr2:111348488-111348538(-),CTCTGACTCTTGACCTGGTCTGGTCACTTGCTGTGGTTGTGTGTGA...,GGTGTCACACACAACCACAGCAAGTGACCAGACCAGGTCAAGAGTC...,0.54,2,0.52,0.56,True,CTCTGACTCTTGACCTGGTCTGGTC,ACTTGCTGTGGTTGTGTGTGACACC
3,chr2,111289808,111289858,MIR4435-2HG_7_93943,.,-,chr2:111289808-111289858(-),CTTCCTACCTGGTAGTAGCACTGACATCCTCTTCATCCACTGGAAG...,GAAGCTTCCAGTGGATGAAGAGGATGTCAGTGCTACTACCAGGTAG...,0.50,2,0.48,0.52,True,CTTCCTACCTGGTAGTAGCACTGAC,ATCCTCTTCATCCACTGGAAGCTTC
4,chr2,111289810,111289860,MIR4435-2HG_7_93945,.,-,chr2:111289810-111289860(-),GCCTTCCTACCTGGTAGTAGCACTGACATCCTCTTCATCCACTGGA...,AGCTTCCAGTGGATGAAGAGGATGTCAGTGCTACTACCAGGTAGGA...,0.52,2,0.48,0.56,True,GCCTTCCTACCTGGTAGTAGCACTG,ACATCCTCTTCATCCACTGGAAGCT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29184,chr2,111422162,111422212,MIR4435-2HG_4_214390,.,-,chr2:111422162-111422212(-),AGGCTGCCTTACTAGAAGCAGAGTCAGCCATGGGGGGGTGAGAACA...,CTTTTGTTCTCACCCCCCCATGGCTGACTCTGCTTCTAGTAAGGCA...,0.54,7,0.56,0.52,True,AGGCTGCCTTACTAGAAGCAGAGTC,AGCCATGGGGGGGTGAGAACAAAAG
29185,chr2,111422166,111422216,MIR4435-2HG_2_214394,.,-,chr2:111422166-111422216(-),GGAAAGGCTGCCTTACTAGAAGCAGAGTCAGCCATGGGGGGGTGAG...,TGTTCTCACCCCCCCATGGCTGACTCTGCTTCTAGTAAGGCAGCCT...,0.56,7,0.60,0.52,True,GGAAAGGCTGCCTTACTAGAAGCAG,AGTCAGCCATGGGGGGGTGAGAACA
29186,chr2,111422158,111422208,MIR4435-2HG_4_214386,.,-,chr2:111422158-111422208(-),TGCCTTACTAGAAGCAGAGTCAGCCATGGGGGGGTGAGAACAAAAG...,CACACTTTTGTTCTCACCCCCCCATGGCTGACTCTGCTTCTAGTAA...,0.52,7,0.52,0.52,True,TGCCTTACTAGAAGCAGAGTCAGCC,ATGGGGGGGTGAGAACAAAAGTGTG
29187,chr2,111282309,111282359,MIR4435-2HG_1_74537,.,-,chr2:111282309-111282359(-),GGGTGGGGGGGCCTTCTTGCTTGCTAGTGACAATCTACACCAACCA...,GTGCTGGTTGGTGTAGATTGTCACTAGCAAGCAAGAAGGCCCCCCC...,0.58,7,0.48,0.68,True,GGGTGGGGGGGCCTTCTTGCTTGCT,AGTGACAATCTACACCAACCAGCAC


In [40]:
## We have a lot of probes here - increasing the offset to 1000 bp to space them out
# probe_offset = 1000 # uncomment to make changes to the top-level arguments 

In [41]:
# Select probes by row coordinates instead of a potentially duplicated name.
selected_probes_list = []
df = kmers.copy()

while len(selected_probes_list) < n_desired_probes and not df.empty:
    selected_probe = df.iloc[[0]].copy()
    selected_probes_list.append(selected_probe)

    start = int(selected_probe["start"].iloc[0])
    end = int(selected_probe["end"].iloc[0])

    df = (
        df.loc[
            (df["end"] < start - probe_offset)
            | (df["start"] > end + probe_offset)
        ]
        .reset_index(drop=True)
        .copy()
    )

selected_probes = pd.concat(selected_probes_list, ignore_index=True)

In [42]:
selected_probes_df = pd.concat(selected_probes_list, axis=0).reset_index(drop=True)

In [43]:
selected_probes_df

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS
0,chr2,111414483,111414533,MIR4435-2HG_2_206711,.,-,chr2:111414483-111414533(-),CCTGAGTTGCTGCCAACCTTACAATATCCAGCTCCGTCCTCAGAGC...,TGTGGCTCTGAGGACGGAGCTGGATATTGTAAGGTTGGCAGCAACT...,0.54,2,0.60,0.48,True,CCTGAGTTGCTGCCAACCTTACAAT,ATCCAGCTCCGTCCTCAGAGCCACA
1,chr2,111315351,111315401,MIR4435-2HG_1_107579,.,-,chr2:111315351-111315401(-),GCTCTCCTGTTACTTCCTGTGCTTAATTAAGTGGCTCTGCAGGTGA...,TGAGTCACCTGCAGAGCCACTTAATTAAGCACAGGAAGTAACAGGA...,0.48,2,0.48,0.48,True,GCTCTCCTGTTACTTCCTGTGCTTA,ATTAAGTGGCTCTGCAGGTGACTCA
2,chr2,111348488,111348538,MIR4435-2HG_2_140716,.,-,chr2:111348488-111348538(-),CTCTGACTCTTGACCTGGTCTGGTCACTTGCTGTGGTTGTGTGTGA...,GGTGTCACACACAACCACAGCAAGTGACCAGACCAGGTCAAGAGTC...,0.54,2,0.52,0.56,True,CTCTGACTCTTGACCTGGTCTGGTC,ACTTGCTGTGGTTGTGTGTGACACC


In [44]:
for seq in selected_probes_df['transcript_seq']:
    print(seq)

CCTGAGTTGCTGCCAACCTTACAATATCCAGCTCCGTCCTCAGAGCCACA
GCTCTCCTGTTACTTCCTGTGCTTAATTAAGTGGCTCTGCAGGTGACTCA
CTCTGACTCTTGACCTGGTCTGGTCACTTGCTGTGGTTGTGTGTGACACC


And we are done! We have now selected three potential probes for our gene.

If our probes are meant to be split, we can additionally generate these columns:

In [45]:
if split_nt is not None: 
    ## Make split probes (and add adapters if provided)
    selected_probes_df['probe_seq_LHS'] = [(LHS_pref + seq[0:split_nt] + LHS_suff) for seq in selected_probes_df['probe_seq']]
    selected_probes_df['probe_seq_RHS'] = [(RHS_pref +seq[split_nt: probe_length] + RHS_suff) for seq in selected_probes_df['probe_seq']]

In [46]:
## Also add gene_ID for completeness
selected_probes_df['gene_ID'] = gene_ID

In [47]:
## Export selected probes as dataframe:
selected_probes_df.to_csv((out_dir + 'kmers_selected_probes.csv'))

We always recommend additionally performing a manual BLAST of these probe sequences to make sure that there are no off-target effects.

In [48]:
selected_probes_df

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS,probe_seq_LHS,probe_seq_RHS,gene_ID
0,chr2,111414483,111414533,MIR4435-2HG_2_206711,.,-,chr2:111414483-111414533(-),CCTGAGTTGCTGCCAACCTTACAATATCCAGCTCCGTCCTCAGAGC...,TGTGGCTCTGAGGACGGAGCTGGATATTGTAAGGTTGGCAGCAACT...,0.54,2,0.60,0.48,True,CCTGAGTTGCTGCCAACCTTACAAT,ATCCAGCTCCGTCCTCAGAGCCACA,CCTTGGCACCCGAGAATTCCATGTGGCTCTGAGGACGGAGCTGGAT,/5Phos/ATTGTAAGGTTGGCAGCAACTCAGGCCCATATAAGAAA,MIR4435-2HG
1,chr2,111315351,111315401,MIR4435-2HG_1_107579,.,-,chr2:111315351-111315401(-),GCTCTCCTGTTACTTCCTGTGCTTAATTAAGTGGCTCTGCAGGTGA...,TGAGTCACCTGCAGAGCCACTTAATTAAGCACAGGAAGTAACAGGA...,0.48,2,0.48,0.48,True,GCTCTCCTGTTACTTCCTGTGCTTA,ATTAAGTGGCTCTGCAGGTGACTCA,CCTTGGCACCCGAGAATTCCATGAGTCACCTGCAGAGCCACTTAAT,/5Phos/TAAGCACAGGAAGTAACAGGAGAGCCCCATATAAGAAA,MIR4435-2HG
2,chr2,111348488,111348538,MIR4435-2HG_2_140716,.,-,chr2:111348488-111348538(-),CTCTGACTCTTGACCTGGTCTGGTCACTTGCTGTGGTTGTGTGTGA...,GGTGTCACACACAACCACAGCAAGTGACCAGACCAGGTCAAGAGTC...,0.54,2,0.52,0.56,True,CTCTGACTCTTGACCTGGTCTGGTC,ACTTGCTGTGGTTGTGTGTGACACC,CCTTGGCACCCGAGAATTCCAGGTGTCACACACAACCACAGCAAGT,/5Phos/GACCAGACCAGGTCAAGAGTCAGAGCCCATATAAGAAA,MIR4435-2HG
